### **Processing of particulate absorption (PerkinElmer)**

This notebook demonstrates how to use the ProLab module to process particulate absorption data acquired with the PerkinElmer spectrophotometer, including spectral break correction.

**Author:** Bruno Rech

**Last update:** 16 Oct 2026

---

#### **Dependencies**

Make sure that all libraries are installed. We recommend creating a dedicated environment; follow the instructions in the [read-me](https://github.com/b-rech/prolab) file.

In [ ]:
import prolab as pl
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import os

#### **Open data**

Let us start by setting the working directory:

In [ ]:
working_dir = Path('C:/Users/brech/Desktop/repositories/prolab/examples')
os.chdir(working_dir)

To each set of measurements, you must have a folder; in our example, it is called `TR_20260504`. Inside this folder, there are:

1. another folder called `raw`, in which **only** the measurement files are stored;
2. a CSV file containing each station's filtered volume and clearance diameter.

All files in `raw` must follow the same naming convention. We provide the "explanation" of what each part of the file names mean to the `pl.read_files` function through the argument `format_string`.

Do not forget to ensure that the volumes are in m³, and, the diameter, in m.

To check a function's documentation, run `help(function_name)`.

In [ ]:
# Path to the data folder
data_path = Path('data/TR_20260504')

# Open volume and diameter information
vol_diameter = pd.read_csv(data_path / 'vol_diameter.csv',
                           sep=';', decimal='.', index_col=0)

# Convert units
vol_diameter['volume'] /= 1e6     # mL to m³
vol_diameter['diameter'] /= 1000  # mm to m

# Load dataset - for more details, run help(pl.read_files)
# In the format string, 'rmode' refers to transmittance/reflectance, whereas
# 'msr' refers to total or depigmented measurements.
raw = pl.read_files(
    path=data_path / 'raw',
    instrument='perkinelmer',
    format_string='{camp}_{pattern}_{id}_{rmode}_{msr}',
    blank_pattern='REF',
    sample_pattern='AMO',
    depig_pattern='EXT',
    decimal='.'
    )

# Create a Spectra object
raw_spec = pl.Spectra(raw)

The CSV of volumes and diameters must follow this format:

In [ ]:
vol_diameter

Now that the samples and blanks are open, we may also open the "standard" blank curves. They were obtained from averaging many blank measurements from various days, after the removal of ouliers:

In [ ]:
standard_blanks = pd.read_csv('data/standard_blanks_perkinelmer.csv',
                              sep=';', decimal='.', index_col=0)
standard_blanks.columns = standard_blanks.columns.map(int)

#### **Break correction**

Before anything, we need to assess the occurrence of breaks in the measurements and correct them before advancing with the processing. The procedure follows some steps:

1. The break correction starts when the cell below is executed;
2. The breaks are usually around 4 wavelengths highlighted in the plots;
3. A letter from A do D is assigned to each break region;
4. Looking at the plot, you must indicate the letters in which break occurs (all together), and then press enter;
5. In case of no breaks, just press enter (you can also do that to skip air curves);
6. In case of breaks, a new plot with the raw and the corrected curves will be displayed.

You can use the argument `dpi` the control the plot size (default is 200 DPI).

In [ ]:
# Run correction
spec = raw_spec.correction.treat_breaks(inline=True, dpi=200)

Once the spectra are corrected, we must save them:

In [ ]:
# Save corrected spectra
spec.raw_data.to_csv(data_path / 'break_corrected.csv',
                     index=True, decimal='.', sep=';')

In case you have already performed the correction and saved it, and want to resume the processing after closing this notebook, you can skip the two cells above and open the corrected spectra directly:

In [ ]:
spec = pd.read_csv(data_path / 'break_corrected.csv',
                       sep=';', decimal='.', index_col=0)
spec = pl.Spectra(spec)

#### **Check blanks**

Now that our data is break-corrected, we may want to compare the blanks with the "standard" curves to check consistency.

First, lets isolate our blanks an plot them:

In [ ]:
# Get only blanks
blanks = spec.raw_spectra.loc[spec.is_blank]
blanks.columns = blanks.columns.map(int)
blanks_to_plot = blanks.filter(regex='\d').transpose()
blanks.insert(0, 'rmode', spec.rmode.loc[spec.is_blank])

# Get mean blanks
mean_blanks = blanks.groupby('rmode').mean()
mean_blanks.columns = mean_blanks.columns.map(int)

# Create figure
fig, ax = plt.subplots(figsize=(5, 3), dpi=200)

# Plot the blanks
blanks_to_plot.plot(legend=False, color='gray', lw=.5, ax=ax)

# Plot the mean blanks
mean_blanks.transpose().plot(legend=False, color='black', ax=ax)

# Plot the standard blanks
standard_blanks.transpose().plot(legend=False, color='red', ax=ax)
# Configure the plot
ax.set(xlabel='Wavelength (nm)', ylabel='Absorbance (AU)')
ax.grid(which='both', lw=.25, color='black')
sns.despine()
plt.show()

In this case, although the blanks vary a lot, their means are close to the standard blank. We could also separate between total and depigmented blanks, but in this example we are not doing it. The function for calculating absorption allows us to choose between splitting the blanks into total and depigmented.

#### **Calculate absorption**

Finally, after corrections and verifications, we can retrieve the absorption curves:

In [ ]:
# Retrieve curves (check the documentation for more details on the parameters)
spec.particulate.get_absorption(
    vol_diameter=vol_diameter,
    percentual=False,
    type_measurement='absorbance',
    wl_offset=800,
    use_tau=True,
    trans_pattern='T',
    wl_range=None,
    plot_raw=True,
    split_total_depig_blank=False,
    custom_blanks=None
)

After that, we can take a look at the absorption curves:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(6, 2), dpi=200, sharey=True)

# Total absorption 
spec.part_abs_total.transpose().plot(ax=axes[0], legend=False, lw=.5)

# Detritus absorption
spec.part_abs_depig.transpose().plot(ax=axes[1], lw=.5)

# Phytoplankton absorption
spec.part_abs_total.subtract(spec.part_abs_depig).transpose().plot(ax=axes[2],
                                                                           lw=.5,
                                                                           legend=False)

# configure the plots
axes[1].legend(frameon=False, fontsize=6, title='Station')
sns.despine()

for ax, title in zip(axes, ['Total', 'Detritus', 'Phytoplankton']):
    ax.set(xlabel='Wavelength (nm)', ylabel=r'Absorption $\mathrm{(m^{-1})}$')
    ax.set_title(title)

plt.show()

Finally, let us save the curves:

In [ ]:
# Total absorption
spec.part_abs_total.to_csv(data_path / 'particulate_abs_total.csv',
                           index=True, decimal='.', sep=';')

# Detritus absorption
spec.part_abs_depig.to_csv(data_path / 'particulate_abs_depig.csv',
                               index=True, decimal='.', sep=';')

# Phytoplankton absorption
spec.part_abs_total.subtract(spec.part_abs_depig).to_csv(
    data_path / 'particulate_abs_phyto.csv',
    index=True, decimal='.', sep=';'
    )